In [1]:

# 1. Импорт библиотек

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor



# 2. Загрузка данных

sheet_id = "1q-nbWuFrfrIBMXmZfNW78N3bx5v60Vb9"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(url)



# 3. Очистка данных


# удаляю служебный столбец
df = df.drop(columns=['Unnamed: 0'])

# заполняю пропуски
df = df.fillna(df.median(numeric_only=True))

# удаляю low variance признаки
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)
X_temp = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])

selector.fit(X_temp)

low_var_cols = X_temp.columns[~selector.get_support()]
df = df.drop(columns=low_var_cols)



# 4. Подготовка данных


X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = df['IC50, mM']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



# 5. Linear Regression

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("MSE:", mean_squared_error(y_test, y_pred_lr))
print("R2:", r2_score(y_test, y_pred_lr))



# 6. Decision Tree

tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

print("\nDecision Tree")
print("MAE:", mean_absolute_error(y_test, y_pred_tree))
print("MSE:", mean_squared_error(y_test, y_pred_tree))
print("R2:", r2_score(y_test, y_pred_tree))



# 7. Random Forest

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\nRandom Forest")
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("MSE:", mean_squared_error(y_test, y_pred_rf))
print("R2:", r2_score(y_test, y_pred_rf))

Linear Regression
MAE: 264.74316749772936
MSE: 235031.805076678
R2: 0.29538208156868606

Decision Tree
MAE: 235.78402299391493
MSE: 226448.97279272444
R2: 0.3211131413126702

Random Forest
MAE: 235.17510126342142
MSE: 195870.61662725438
R2: 0.4127860860161989


### Что получилось

**Linear Regression**
- MAE: **264.7**
- MSE: **235031.8**
- R2: **0.295**

**Decision Tree**
- MAE: **235.8**
- MSE: **226449.0**
- R2: **0.321**

**Random Forest**
- MAE: **235.2**
- MSE: **195870.6**
- R2: **0.413**

### Вывод

- Линейная модель даёт самый слабый результат  
- Decision Tree улучшает качество, но не сильно  
- Лучший результат сейчас у **Random Forest**  

 пока что именно Random Forest выглядит как основная модель для IC50  

Следующее:
- добавить **подбор гиперпараметров Random Forest**

In [3]:
# Подбор гиперпараметров Random Forest (IC50)
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Лучшие параметры:", random_search.best_params_)

best_rf = random_search.best_estimator_
y_pred = best_rf.predict(X_test)

print("\nПосле подбора:")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

Лучшие параметры: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_depth': None}

После подбора:
MAE: 236.7960878530024
MSE: 198566.05039100704
R2: 0.4047052608390904


### Что делаю

- Подбираю параметры Random Forest через `RandomizedSearchCV`
- Перебираю:
   `n_estimators`
   `max_depth`
   `min_samples_split`
   `min_samples_leaf`

### Что получилось

Лучшие параметры:
- `n_estimators = 200`
- `max_depth = None`
- `min_samples_split = 10`
- `min_samples_leaf = 2`

После подбора:
- **MAE: 236.8**
- **MSE: 198566.1**
- **R2: 0.405**

### Вывод

- Подбор параметров не улучшил модель
- Базовый Random Forest был лучше:
   было **R2 = 0.413**
   стало **R2 = 0.405**

 в качестве финальной модели для IC50 оставляю **обычный Random Forest**

### Финальный вывод по IC50

- Лучшая модель: **Random Forest**
- Качество:
   **R2 ≈ 0.41**
   ошибка заметно ниже, чем у других моделей  

- Linear Regression не справилась  
- Decision Tree дала небольшой прирост  
- Подбор гиперпараметров не дал улучшения  

→ фиксирую Random Forest как итоговую модель  

Следующее:
- создаю файл **model_cc50.ipynb** и повторяю процесс